In [1]:
import os
import sys
import time

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.regularizers import (
    FastFixPhiRegularizer, DecorrelateWithOtherPhiRegularizer, DecorrelateWithOtherPhiRegularizer2
)
from topnum.scores.intratext_coherence_score import (
    IntratextCoherenceScore,
    ComputationMethod,
    WordTopicRelatednessType,
)
from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED, init_plsa

In [4]:
import artm
from artm import ARTM, Dictionary

import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

from topicnet.cooking_machine.models.topic_model import ARTM_NINE
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.viewers.top_documents_viewer import TopDocumentsViewer
from topicnet.viewers.top_tokens_viewer import TopTokensViewer
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    count_vocab_size,
    init_model,
)
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    modality_weight_rel2abs,
    transform_regularizer,
)


import numpy as np
import pandas as pd
from pandas import DataFrame
from scipy.spatial.distance import cdist

import os
import tempfile
import warnings
from copy import deepcopy
from typing import Dict, List, Optional

In [5]:
from IPython.display import display, display_html

In [6]:
DATA_FOLDER_PATH = '/data_mil/shared/CompressaAI/iterative/data/noow'

In [7]:
! ls $DATA_FOLDER_PATH

_20_Newsgroups.csv	       MKB_10_NOOW__internals
_20_Newsgroups__internals      Post_Science__internals
20_Newsgroups__internals       Post_Science_NOOW.csv
20_Newsgroups_NOOW.csv	       Post_Science_NOOW_fixed.csv
20_Newsgroups_NOOW__internals  Post_Science_NOOW_fixed__internals
_Lenta.csv		       Post_Science_NOOW__internals
MKB_10__internals	       WikiRef_220_NOOW.csv
MKB_10_NOOW.csv


In [8]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/Post_Science_NOOW_fixed.csv',
)

dataset.get_possible_modalities()

{'@word'}

In [9]:
MAIN_MODALITY = '@word'

In [10]:
dataset._data.head()

,id,raw_text,vw_text
id,,,
29998.txt,29998.txt,материал отрицательный показатель преломление ...,29998.txt |@word материал отрицательный показа...
7770.txt,7770.txt,культурный код экономика экономист александр а...,7770.txt |@word культурный код экономика эконо...
32230.txt,32230.txt,faq наука третий класс факт эксперимент резуль...,32230.txt |@word faq наука третий класс факт э...
27293.txt,27293.txt,обрушение волна поверхность жидкость математик...,27293.txt |@word обрушение волна поверхность ж...
481.txt,481.txt,существовать ли суперсимметрия мир элементарны...,481.txt |@word существовать ли суперсимметрия ...


In [11]:
dataset._data.shape

(3446, 3)

In [12]:
dictionary = dataset.get_dictionary()

print(dictionary)

for modality in dataset.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=41d87d50-a85d-465c-9786-07aab906aa95, num_entries=82162)


In [13]:
print(dictionary)

artm.Dictionary(name=41d87d50-a85d-465c-9786-07aab906aa95, num_entries=82162)


In [14]:
dictionary.filter(min_df=5, max_df_rate=0.5)

artm.Dictionary(name=41d87d50-a85d-465c-9786-07aab906aa95, num_entries=19537)

In [15]:
dataset._cached_dict = dictionary

In [16]:
dataset.get_dictionary()

artm.Dictionary(name=41d87d50-a85d-465c-9786-07aab906aa95, num_entries=19537)

In [17]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [18]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 6.54 s, sys: 236 ms, total: 6.78 s
Wall time: 6.7 s


In [19]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [20]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [21]:
def view_model(
        topic_model,
        dataset,
        num_top_tokens: int = 5,
        top_tokens_method: str = 'phi',
        num_topics: Optional[int] = 5,  # we do not want to fill the whole .ipynb notebook with topics...
        ):
    top_tok_viewer = TopTokensViewer(
        topic_model, num_top_tokens=num_top_tokens, method=top_tokens_method
    )
    top_doc_viewer = TopDocumentsViewer(topic_model, dataset=dataset)
    top_docs = top_doc_viewer.view()

    if num_topics is None:
        num_topics = len(topic_model.topic_names)

    for topic_name in topic_model.topic_names[:num_topics]:
        topic_top_toks = top_tok_viewer.to_html(topic_names=[topic_name])
        topic_top_docs = top_docs[topic_name]
        display_html(topic_top_toks, raw=True)
        display(topic_top_docs)

In [22]:
parent_model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=10,
    seed=2024,
)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



In [47]:
%%time

parent_model._fit(
    dataset.get_batch_vectorizer(),
    num_iterations=10,
)

CPU times: user 14 s, sys: 0 ns, total: 14 s
Wall time: 6.03 s


In [48]:
view_model(parent_model, dataset)

topic_0 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 клетка 
 0.009860 
 
 
 мозг 
 0.006920 
 
 
 ген 
 0.005980 
 
 
 организм 
 0.004520 
 
 
 происходить 
 0.003810

{'3230.txt': 0.99999046,
 '1145.txt': 0.99990153,
 '1142.txt': 0.9998315,
 '683.txt': 0.9997149,
 '803.txt': 0.999634,
 '1603.txt': 0.99962556,
 '2559.txt': 0.9994853,
 '736.txt': 0.9992607,
 '1570.txt': 0.99866533,
 '1155.txt': 0.99850446}

topic_1 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 книга 
 0.007100 
 
 
 понятие 
 0.006040 
 
 
 общество 
 0.005400 
 
 
 являться 
 0.005340 
 
 
 культура 
 0.005240

{'261.txt': 0.9999571,
 '2158.txt': 0.99360806,
 '1744.txt': 0.97676593,
 '1389.txt': 0.9752322,
 '1714.txt': 0.9726094,
 '2600.txt': 0.968191,
 '1323.txt': 0.9578743,
 '550.txt': 0.94791764,
 '449.txt': 0.9462596,
 '2583.txt': 0.9447587}

topic_2 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 язык 
 0.029660 
 
 
 слово 
 0.015680 
 
 
 говорить 
 0.007200 
 
 
 текст 
 0.005980 
 
 
 русский 
 0.004390

{'1765.txt': 0.9994957,
 '2143.txt': 0.9990222,
 '2708.txt': 0.99848783,
 '3.txt': 0.99787784,
 '760.txt': 0.9945842,
 '54.txt': 0.9927408,
 '719.txt': 0.99152446,
 '1114.txt': 0.9868923,
 '2914.txt': 0.9859373,
 '2495.txt': 0.98394877}

topic_3 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 город 
 0.010730 
 
 
 задача 
 0.006010 
 
 
 caption 
 0.004920 
 
 
 метр 
 0.003790 
 
 
 дать 
 0.003670

{'2655.txt': 0.95536774,
 '603.txt': 0.94023144,
 '2231.txt': 0.908404,
 '1949.txt': 0.89367205,
 '375.txt': 0.8713503,
 '2083.txt': 0.8436019,
 '1494.txt': 0.8161912,
 '2463.txt': 0.7955459,
 '942.txt': 0.79392004,
 '1057.txt': 0.7914539}

topic_4 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 история 
 0.008650 
 
 
 книга 
 0.007780 
 
 
 говорить 
 0.007720 
 
 
 слово 
 0.006750 
 
 
 фильм 
 0.005710

{'433.txt': 0.99390864,
 '1598.txt': 0.9797057,
 '2116.txt': 0.9593311,
 '1434.txt': 0.95547,
 '3283.txt': 0.94927233,
 '3056.txt': 0.9439146,
 '2954.txt': 0.9436736,
 '2944.txt': 0.93279225,
 '3080.txt': 0.9312259,
 '2378.txt': 0.92740524}

In [49]:
model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=10,
    seed=42,
)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



In [50]:
regularizer = FastFixPhiRegularizer(
    name='fix',
    parent_model=parent_model._model,
    topic_names=['topic_0', 'topic_1', 'topic_2'],
)

In [52]:
%%time

model._fit(
    dataset.get_batch_vectorizer(),
    num_iterations=10,
    custom_regularizers={
        regularizer.name: regularizer,
    }
)

CPU times: user 17.6 s, sys: 0 ns, total: 17.6 s
Wall time: 9.21 s


In [53]:
view_model(model, dataset)

topic_0 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 клетка 
 0.009860 
 
 
 мозг 
 0.006920 
 
 
 ген 
 0.005980 
 
 
 организм 
 0.004520 
 
 
 происходить 
 0.003810

{'3230.txt': 0.99996674,
 '1142.txt': 0.9997461,
 '1145.txt': 0.9997289,
 '2559.txt': 0.99952406,
 '1603.txt': 0.99948597,
 '683.txt': 0.9994554,
 '736.txt': 0.9987189,
 '1155.txt': 0.99852514,
 '1570.txt': 0.99852026,
 '756.txt': 0.9972416}

topic_1 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 книга 
 0.007100 
 
 
 понятие 
 0.006040 
 
 
 общество 
 0.005400 
 
 
 являться 
 0.005340 
 
 
 культура 
 0.005240

{'261.txt': 0.9999601,
 '2158.txt': 0.9986314,
 '575.txt': 0.9977152,
 '2732.txt': 0.99255204,
 '1744.txt': 0.9914585,
 '1389.txt': 0.96807724,
 '1714.txt': 0.9643121,
 '1323.txt': 0.95993567,
 '550.txt': 0.9585386,
 '3040.txt': 0.9529556}

topic_2 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 язык 
 0.029660 
 
 
 слово 
 0.015680 
 
 
 говорить 
 0.007200 
 
 
 текст 
 0.005980 
 
 
 русский 
 0.004390

{'1765.txt': 0.9990342,
 '3.txt': 0.9986576,
 '2708.txt': 0.99770737,
 '760.txt': 0.9962234,
 '719.txt': 0.9921972,
 '54.txt': 0.99025005,
 '2914.txt': 0.9857618,
 '58.txt': 0.98354626,
 '1055.txt': 0.98330015,
 '2495.txt': 0.9831579}

topic_3 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 страна 
 0.008430 
 
 
 россия 
 0.008380 
 
 
 государство 
 0.005540 
 
 
 университет 
 0.004850 
 
 
 должный 
 0.004340

{'1853.txt': 0.99625605,
 '1209.txt': 0.9961494,
 '1405.txt': 0.9821817,
 '1957.txt': 0.96341294,
 '889.txt': 0.9493567,
 '979.txt': 0.94332564,
 '1621.txt': 0.94091934,
 '2268.txt': 0.93985444,
 '2248.txt': 0.93924767,
 '791.txt': 0.9326045}

topic_4 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 задача 
 0.007990 
 
 
 дать 
 0.006860 
 
 
 система 
 0.006190 
 
 
 работа 
 0.005820 
 
 
 исследование 
 0.004970

{'1277.txt': 0.9993375,
 '2357.txt': 0.99749625,
 '1940.txt': 0.99173695,
 '2217.txt': 0.9886633,
 '18.txt': 0.97835696,
 '1793.txt': 0.97121596,
 '3404.txt': 0.97027576,
 '1978.txt': 0.970227,
 '1867.txt': 0.9631704,
 '2652.txt': 0.96316534}

In [89]:
model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=10,
    seed=42,
)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



In [90]:
model.get_phi().shape

(19186, 10)

In [57]:
%%time

model._fit(
    dataset.get_batch_vectorizer(),
    num_iterations=NUM_ITERATIONS
)

CPU times: user 29.7 s, sys: 0 ns, total: 29.7 s
Wall time: 13 s


In [58]:
other_phi = model._model.get_phi()[['topic_0', 'topic_1', 'topic_2']]
other_phi = deepcopy(other_phi)

regularizer = DecorrelatorWithOtherPhiRegularizer(
    name='ext_decorr', tau=25,  # 1e5
    topic_names=['topic_0', 'topic_1', 'topic_2'],
    other_phi=other_phi
)

In [60]:
%%time

model._fit(
    dataset.get_batch_vectorizer(),
    num_iterations=NUM_ITERATIONS,
    custom_regularizers={
        regularizer.name: regularizer,
    }
)

CPU times: user 34.5 s, sys: 523 µs, total: 34.5 s
Wall time: 18.2 s


In [62]:
model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=10,
    seed=42,
)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



In [63]:
%%time

model._fit(
    dataset.get_batch_vectorizer(),
    num_iterations=NUM_ITERATIONS
)

CPU times: user 29.7 s, sys: 0 ns, total: 29.7 s
Wall time: 13 s


In [64]:
other_phi = model._model.get_phi()[['topic_0', 'topic_1', 'topic_2']]
other_phi = deepcopy(other_phi)

regularizer = DecorrelatorWithOtherPhiRegularizer2(
    name='ext_decorr', tau=25,  # 1e5
    topic_names=['topic_0', 'topic_1', 'topic_2'],
    other_phi=other_phi
)

In [65]:
%%time

model._fit(
    dataset.get_batch_vectorizer(),
    num_iterations=NUM_ITERATIONS,
    custom_regularizers={
        regularizer.name: regularizer,
    }
)

CPU times: user 34.5 s, sys: 0 ns, total: 34.5 s
Wall time: 18.2 s


In [24]:
NUM_TOPICS = 20  # vary
MAX_NUM_TRAINS = 20
NUM_ITERATIONS = 20
NUM_TOP_TOKENS = 20

In [34]:
def fit_and_compute_scores(model, dataset, custom_regularizers=None):
    print(custom_regularizers)

    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS, custom_regularizers=custom_regularizers)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}


    
    # intra1 = IntratextCoherenceScore(
    #     name='toplen_pwt',
    #     data=dataset,
    #     computation_method=ComputationMethod.SEGMENT_LENGTH,
    #     word_topic_relatedness=WordTopicRelatednessType.PWT,
    #     should_compute=False,  # only on last iter
    # )
    intra2 = IntratextCoherenceScore(
        name='toplen_ptw',
        data=dataset,
        computation_method=ComputationMethod.SEGMENT_LENGTH,
        word_topic_relatedness=WordTopicRelatednessType.PTW,
        should_compute=False,
    )
    # intra3 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     should_compute=False,
    # )
    # intra3_w4 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     window=4,
    #     should_compute=False,
    # )

    intra_topic_coherences = dict()

    for intra in [intra2]:  # [intra2, intra3_w4]:  #[intra1, intra2, intra3]:
        # print(f'\nComputing "{intra._name}"...')

        current_intra_topic_coherences = intra.compute(model)

        assert all(v is not None for v in current_intra_topic_coherences.values())

        _values = current_intra_topic_coherences.values()

        current_intra_topic_coherences = {
            i: current_intra_topic_coherences[t]  # if v is not None else 0.0
            for i, t in enumerate(target_topic_names)
        }

        assert all(abs(x - y) <= 1e-6 for x, y in zip(_values, current_intra_topic_coherences.values())), (_values, current_intra_topic_coherences.values())  # "sorted" Python dicts
        
        intra_topic_coherences[f'topic_coherences_{intra._name}'] = current_intra_topic_coherences

        value = float(np.median(list(current_intra_topic_coherences.values())))
        score_values[intra._name] = value

        # print(f'Result by topic: {current_intra_topic_coherences}.')

    

    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    
    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
        **intra_topic_coherences,
    }

In [26]:
# HIGH_COHERENCE_THRESHOLD = 1.0014147388051453
# LOW_COHERENCE_THRESHOLD = 0.49633189795078303

def is_good(coherence):
    return 2.3394925682940233 <= coherence

def is_bad(coherence):
    return coherence <= 1.7938763485672071

## Test

In [86]:
model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS,
    seed=2024,
)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



In [87]:
result = fit_and_compute_scores(model, dataset)

In [88]:
result['topic_coherences']

{0: 0.8022360201373641,
 1: 0.6519442202687643,
 2: 0.6226721565500578,
 3: 0.9588649137362095,
 4: 0.704356727143289,
 5: 0.8469110127774976,
 6: 0.5129234479663343,
 7: 0.38927422916278404,
 8: 0.6261746122878845,
 9: 0.9762188973210647,
 10: 0.4949569134930998,
 11: 1.04632016562861,
 12: 0.5007718039696671,
 13: 0.4310834913251312,
 14: 0.9270152334414302,
 15: 0.44998837489550614,
 16: 0.7694945577901475,
 17: 0.9222548869825914,
 18: 0.39456919601067664,
 19: 0.6681510538091538}

In [31]:
result['topic_coherences_toplen_ptw']

{0: 2.0447393515755286,
 1: 1.9344117556298146,
 2: 1.8710572784293265,
 3: 1.6490425036834446,
 4: 1.6849669636564772,
 5: 2.1557139103148493,
 6: 2.291794130689408,
 7: 2.4785042484777016,
 8: 2.0838985824554115,
 9: 3.934897813806161,
 10: 1.9035606434296308,
 11: 2.050574940352245,
 12: 2.305221824356423,
 13: 2.689761241973405,
 14: 3.192872091945433,
 15: 1.6788912651575991,
 16: 3.330320519896818,
 17: 1.8832803063554624,
 18: 2.666609155095051,
 19: 2.5383185779412583}

In [89]:
good_topic_indices = [
    t for t, c in result['topic_coherences_toplen_ptw'].items() if is_good(c)  # c >= HIGH_COHERENCE_THRESHOLD
]
bad_topic_indices = [
    t for t, c in result['topic_coherences_toplen_ptw'].items() if is_bad(c)  # c <= LOW_COHERENCE_THRESHOLD
]

phi = model.get_phi()
good_topic_names = [phi.columns[t] for t in good_topic_indices]
bad_topic_names = [phi.columns[t] for t in bad_topic_indices]

In [90]:
phi.columns

Index(['topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5',
       'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11',
       'topic_12', 'topic_13', 'topic_14', 'topic_15', 'topic_16', 'topic_17',
       'topic_18', 'topic_19'],
      dtype='object')

In [91]:
len(good_topic_indices), len(bad_topic_indices)

(5, 5)

In [92]:
good_topic_names, bad_topic_names

(['topic_9', 'topic_13', 'topic_14', 'topic_16', 'topic_18'],
 ['topic_2', 'topic_3', 'topic_4', 'topic_15', 'topic_17'])

In [35]:
phi['topic_9'].sort_values(ascending=False)[:20]

modality  token      
@word     звезда         0.015509
          галактика      0.010965
          вселенная      0.010941
          планета        0.009520
          земля          0.007508
          черный         0.006631
          дыра           0.006557
          объект         0.006386
          система        0.006006
          масса          0.005831
          солнце         0.005696
          большой        0.005560
          излучение      0.005316
          теория         0.005008
          космический    0.004452
          вещество       0.004364
          скорость       0.004358
          волна          0.004207
          энергия        0.004137
          расстояние     0.004018
Name: topic_9, dtype: float32

In [36]:
phi['topic_13'].sort_values(ascending=False)[:20]

modality  token         
@word     система           0.009878
          задача            0.009395
          информация        0.007603
          данные            0.007396
          компьютер         0.007120
          квантовый         0.006845
          сеть              0.006070
          технология        0.005934
          метод             0.005180
          число             0.004906
          модель            0.004695
          использовать      0.004600
          решение           0.004587
          теория            0.004513
          математический    0.004444
          каждый            0.004265
          получать          0.004141
          сделать           0.004130
          проблема          0.004075
          большой           0.004030
Name: topic_13, dtype: float32

In [37]:
phi['topic_14'].sort_values(ascending=False)[:20]

modality  token       
@word     клетка          0.027105
          ген             0.011330
          белок           0.009139
          днк             0.008733
          организм        0.007823
          молекула        0.007763
          система         0.005919
          метод           0.005263
          получать        0.004673
          заболевание     0.004448
          болезнь         0.004336
          процесс         0.004306
          молекулярный    0.004288
          исследование    0.003820
          бактерия        0.003714
          генетический    0.003707
          препарат        0.003589
          мутация         0.003502
          работать        0.003501
          биология        0.003441
Name: topic_14, dtype: float32

In [38]:
phi['topic_16'].sort_values(ascending=False)[:20]

modality  token         
@word     частица           0.017907
          энергия           0.009169
          поле              0.008204
          атом              0.007265
          материал          0.006915
          физика            0.006530
          взаимодействие    0.006339
          теория            0.006301
          электрон          0.005951
          свойство          0.005606
          магнитный         0.005502
          структура         0.005328
          свет              0.004524
          получать          0.004507
          кварк             0.004495
          два               0.004226
          вещество          0.003980
          модель            0.003943
          квантовый         0.003936
          эксперимент       0.003929
Name: topic_16, dtype: float32

In [39]:
phi['topic_18'].sort_values(ascending=False)[:20]

modality  token       
@word     история         0.019035
          книга           0.018425
          лекция          0.013614
          философия       0.013481
          теория          0.011417
          id              0.009876
          мир             0.009109
          исторический    0.009077
          знание          0.008827
          автор           0.008290
          научный         0.007574
          век             0.007142
          современный     0.006296
          философский     0.006121
          прочитывать     0.006053
          философ         0.005850
          м               0.005770
          идея            0.005645
          интересный      0.005514
          писать          0.005468
Name: topic_18, dtype: float32

In [ ]:
###

In [40]:
phi['topic_2'].sort_values(ascending=False)[:20]

modality  token       
@word     ребенок         0.028711
          женщина         0.015895
          жизнь           0.008866
          мужчина         0.008479
          возраст         0.007260
          социальный      0.006318
          семья           0.006269
          отношение       0.005734
          группа          0.005683
          развитие        0.005540
          мать            0.005070
          часто           0.004953
          родитель        0.004936
          взрослый        0.004881
          поведение       0.004822
          исследование    0.004360
          вид             0.004285
          разный          0.004215
          роль            0.003932
          вес             0.003927
Name: topic_2, dtype: float32

In [41]:
phi['topic_3'].sort_values(ascending=False)[:20]

modality  token           
@word     г                   0.009500
          остров              0.007998
          caption             0.005551
          width               0.005076
          idattachment        0.005010
          место               0.004620
          кавказ              0.004383
          находить            0.003625
          город               0.003612
          святилище           0.003516
          вулкан              0.003484
          рис                 0.003413
          дерево              0.003234
          alignaligncenter    0.003085
          два                 0.002975
          находиться          0.002959
          здание              0.002889
          часть               0.002779
          земля               0.002761
          м                   0.002644
Name: topic_3, dtype: float32

In [43]:
phi['topic_4'].sort_values(ascending=False)[:20]

modality  token       
@word     книга           0.016836
          фильм           0.010628
          литература      0.007806
          искусство       0.007619
          культура        0.006847
          автор           0.006474
          текст           0.005646
          кино            0.005625
          произведение    0.004148
          советский       0.003716
          написать        0.003618
          даже            0.003611
          говорить        0.003596
          литературный    0.003590
          русский         0.003481
          читатель        0.003469
          тема            0.003282
          смысл           0.003179
          важный          0.003163
          писать          0.002936
Name: topic_4, dtype: float32

In [44]:
phi['topic_15'].sort_values(ascending=False)[:20]

modality  token     
@word     век           0.007742
          культура      0.005245
          русский       0.005238
          жизнь         0.004939
          история       0.004241
          дело          0.004198
          театр         0.003709
          мир           0.003549
          япония        0.003376
          там           0.003231
          начинать      0.003190
          писать        0.003182
          говорить      0.003132
          ни            0.003001
          фольклор      0.002819
          герой         0.002804
          xix           0.002775
          совершенно    0.002746
          письмо        0.002718
          день          0.002641
Name: topic_15, dtype: float32

In [93]:
phi['topic_17'].sort_values(ascending=False)[:20]

modality  token        
@word     право            0.016237
          век              0.008892
          власть           0.006388
          церковь          0.006309
          римский          0.005824
          король           0.005497
          закон            0.005084
          бог              0.004942
          средневековый    0.004129
          правовой         0.003643
          дело             0.003500
          мир              0.003221
          политический     0.003205
          суд              0.003193
          говорить         0.003105
          император        0.002931
          святой           0.002805
          должный          0.002765
          история          0.002749
          папа             0.002743
Name: topic_17, dtype: float32

In [94]:
fix_regularizer = FastFixPhiRegularizer(
    name='fix',
    parent_model=model._model,
    topic_names=good_topic_names,
)

other_phi = model._model.get_phi()[bad_topic_names]
other_phi = deepcopy(other_phi)
decorr_bad_regularizer = DecorrelateWithOtherPhiRegularizer(
    name='ext_decorr_bad', tau=10 ** 9,  # 25 1e5
    topic_names=bad_topic_names,
    other_phi=other_phi
)

# other_phi = model._model.get_phi()[good_topic_names]
# other_phi = deepcopy(other_phi)
# decorr_good_regularizer = DecorrelateWithOtherPhiRegularizer(
#     name='ext_decorr_good', tau=25,  # 1e5
#     topic_names=good_topic_names,
#     other_phi=other_phi
# )

In [95]:
%%time

model._fit(
    dataset.get_batch_vectorizer(),
    num_iterations=10,
    custom_regularizers={
        fix_regularizer.name: fix_regularizer,
        decorr_bad_regularizer.name: decorr_bad_regularizer,
        decorr_good_regularizer.name: decorr_good_regularizer,
    }
)

CPU times: user 31.5 s, sys: 214 ms, total: 31.7 s
Wall time: 15.5 s


In [96]:
other_phi = model._model.get_phi()[good_topic_names]
other_phi = deepcopy(other_phi)

In [97]:
other_phi.rename(
    columns={n: f'm1_{n}' for n in good_topic_names}, inplace=True
)

In [98]:
other_phi.columns

Index(['m1_topic_9', 'm1_topic_13', 'm1_topic_14', 'm1_topic_16',
       'm1_topic_18'],
      dtype='object')

In [99]:
phi['topic_9'].sort_values(ascending=False)[:10]

modality  token    
@word     звезда       0.015509
          галактика    0.010965
          вселенная    0.010941
          планета      0.009520
          земля        0.007508
          черный       0.006632
          дыра         0.006557
          объект       0.006386
          система      0.006006
          масса        0.005831
Name: topic_9, dtype: float32

In [100]:
other_phi['m1_topic_9'].sort_values(ascending=False)[:10]

звезда       0.015509
галактика    0.010965
вселенная    0.010941
планета      0.009520
земля        0.007508
черный       0.006631
дыра         0.006557
объект       0.006386
система      0.006006
масса        0.005831
Name: m1_topic_9, dtype: float32

In [101]:
phi['topic_13'].sort_values(ascending=False)[:10]

modality  token     
@word     система       0.009878
          задача        0.009395
          информация    0.007603
          данные        0.007396
          компьютер     0.007120
          квантовый     0.006845
          сеть          0.006070
          технология    0.005934
          метод         0.005180
          число         0.004906
Name: topic_13, dtype: float32

In [102]:
other_phi['m1_topic_13'].sort_values(ascending=False)[:10]

система       0.009878
задача        0.009395
информация    0.007603
данные        0.007396
компьютер     0.007120
квантовый     0.006845
сеть          0.006070
технология    0.005934
метод         0.005180
число         0.004906
Name: m1_topic_13, dtype: float32

In [103]:
phi['topic_14'].sort_values(ascending=False)[:10]

modality  token      
@word     клетка         0.027105
          ген            0.011330
          белок          0.009139
          днк            0.008733
          организм       0.007823
          молекула       0.007763
          система        0.005919
          метод          0.005263
          получать       0.004673
          заболевание    0.004448
Name: topic_14, dtype: float32

In [104]:
other_phi['m1_topic_14'].sort_values(ascending=False)[:10]

клетка         0.027105
ген            0.011330
белок          0.009139
днк            0.008733
организм       0.007823
молекула       0.007763
система        0.005919
метод          0.005263
получать       0.004673
заболевание    0.004448
Name: m1_topic_14, dtype: float32

In [105]:
phi['topic_16'].sort_values(ascending=False)[:10]

modality  token         
@word     частица           0.017907
          энергия           0.009169
          поле              0.008204
          атом              0.007265
          материал          0.006915
          физика            0.006530
          взаимодействие    0.006339
          теория            0.006301
          электрон          0.005951
          свойство          0.005606
Name: topic_16, dtype: float32

In [106]:
other_phi['m1_topic_16'].sort_values(ascending=False)[:10]

частица           0.017907
энергия           0.009169
поле              0.008204
атом              0.007265
материал          0.006915
физика            0.006530
взаимодействие    0.006339
теория            0.006301
электрон          0.005951
свойство          0.005606
Name: m1_topic_16, dtype: float32

In [107]:
phi['topic_18'].sort_values(ascending=False)[:10]

modality  token       
@word     история         0.019035
          книга           0.018425
          лекция          0.013614
          философия       0.013481
          теория          0.011417
          id              0.009876
          мир             0.009109
          исторический    0.009077
          знание          0.008827
          автор           0.008290
Name: topic_18, dtype: float32

In [108]:
other_phi['m1_topic_18'].sort_values(ascending=False)[:10]

история         0.019035
книга           0.018426
лекция          0.013613
философия       0.013481
теория          0.011417
id              0.009876
мир             0.009109
исторический    0.009077
знание          0.008827
автор           0.008290
Name: m1_topic_18, dtype: float32

In [109]:
other_phi = model._model.get_phi()[bad_topic_names]
other_phi = deepcopy(other_phi)

In [110]:
other_phi.rename(
    columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
)

In [111]:
other_phi.columns

Index(['m1_topic_2', 'm1_topic_3', 'm1_topic_4', 'm1_topic_15', 'm1_topic_17'], dtype='object')

In [112]:
phi['topic_2'].sort_values(ascending=False)[:10]

modality  token     
@word     ребенок       0.028711
          женщина       0.015895
          жизнь         0.008866
          мужчина       0.008479
          возраст       0.007260
          социальный    0.006318
          семья         0.006269
          отношение     0.005734
          группа        0.005683
          развитие      0.005540
Name: topic_2, dtype: float32

In [113]:
other_phi['m1_topic_2'].sort_values(ascending=False)[:10]

гормон             0.109842
моногамия          0.056662
рецептор           0.041454
манипулирование    0.035776
онлайн             0.030002
ch                 0.029081
макака             0.028345
межгрупповой       0.028162
паттерн            0.025673
evolution          0.022513
Name: m1_topic_2, dtype: float32

In [114]:
phi['topic_3'].sort_values(ascending=False)[:10]

modality  token       
@word     г               0.009500
          остров          0.007998
          caption         0.005551
          width           0.005076
          idattachment    0.005010
          место           0.004620
          кавказ          0.004383
          находить        0.003625
          город           0.003612
          святилище       0.003516
Name: topic_3, dtype: float32

In [115]:
other_phi['m1_topic_3'].sort_values(ascending=False)[:10]

ньютон         0.102022
чашка          0.070023
грунт          0.052151
дискретный     0.047004
bell           0.045358
марсоход       0.034820
арктический    0.027414
облачный       0.024949
новиков        0.023711
магистраль     0.021881
Name: m1_topic_3, dtype: float32

In [116]:
phi['topic_4'].sort_values(ascending=False)[:10]

modality  token       
@word     книга           0.016837
          фильм           0.010628
          литература      0.007805
          искусство       0.007619
          культура        0.006847
          автор           0.006474
          текст           0.005646
          кино            0.005625
          произведение    0.004148
          советский       0.003716
Name: topic_4, dtype: float32

In [117]:
other_phi['m1_topic_4'].sort_values(ascending=False)[:10]

рейтинг             0.117247
поляризация         0.085163
крыса               0.040936
свертывание         0.037808
смоделировать       0.031787
денис               0.026396
кибернетика         0.023689
абстракция          0.022770
кеннет              0.021054
совершенствовать    0.020459
Name: m1_topic_4, dtype: float32

In [118]:
phi['topic_15'].sort_values(ascending=False)[:10]

modality  token   
@word     век         0.007742
          культура    0.005245
          русский     0.005238
          жизнь       0.004939
          история     0.004241
          дело        0.004198
          театр       0.003709
          мир         0.003549
          япония      0.003376
          там         0.003231
Name: topic_15, dtype: float32

In [119]:
other_phi['m1_topic_15'].sort_values(ascending=False)[:10]

тонна           0.075388
батарея         0.055941
шаманский       0.049776
хрущев          0.042384
истощение       0.032669
считывать       0.030893
определнные     0.023773
николаев        0.021845
обогащаться     0.021608
неодинаковый    0.021372
Name: m1_topic_15, dtype: float32

In [120]:
phi['topic_17'].sort_values(ascending=False)[:10]

modality  token        
@word     право            0.016237
          век              0.008892
          власть           0.006388
          церковь          0.006309
          римский          0.005824
          король           0.005497
          закон            0.005084
          бог              0.004942
          средневековый    0.004129
          правовой         0.003643
Name: topic_17, dtype: float32

In [121]:
other_phi['m1_topic_17'].sort_values(ascending=False)[:10]

регуляция      0.191901
алмаз          0.138192
суверенитет    0.115731
ячейка         0.087775
гоббс          0.086119
радикал        0.057153
неживой        0.048923
аккумулятор    0.045945
state          0.031411
актор          0.030723
Name: m1_topic_17, dtype: float32

In [27]:
def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [28]:
NUM_TRAINS = 3
TOPIC_INDICES = list(range(NUM_TOPICS))

In [29]:
TOPIC_INDICES

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [170]:
prev_results = dict()
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer
DECORRELATION_TAUS = [10, 100, 1000, 10000, 1e5, 1e6, 1e7]


for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    prev_results[key] = []
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.ARTM,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={
                'decorrelation_tau': 0.01,  # best values
                'smooth_bcg_tau': 0.05,
                'sparse_sp_tau': -0.05,
            }
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        result = fit_and_compute_scores(model, dataset)
        prev_results[key].append(result)

        
        good_topic_indices = [
            t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
        ]
        bad_topic_indices = [
            t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
        ]
        not_good_topic_indices = [
            t for t in TOPIC_INDICES if t not in good_topic_indices
        ]
        
        phi = model.get_phi()
        good_topic_names = [phi.columns[t] for t in good_topic_indices]
        bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
        not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]


        assert len(good_topic_names) > 0
        assert len(bad_topic_names) > 0

        print(len(good_topic_names), len(bad_topic_names), len(not_good_topic_names))


        fix_regularizer = FastFixPhiRegularizer(
            name='fix',
            parent_model=model._model,
            topic_names=good_topic_names,
        )
        
        bad_phi = model._model.get_phi()[bad_topic_names]
        bad_phi = deepcopy(bad_phi)
        decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
            name='ext_decorr_bad', tau=decorrelation_tau,
            topic_names=not_good_topic_names,
            other_phi=bad_phi
        )
        
        good_phi = model._model.get_phi()[good_topic_names]
        good_phi = deepcopy(good_phi)
        decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
            name='ext_decorr_good', tau=decorrelation_tau,
            topic_names=not_good_topic_names,
            other_phi=good_phi
        )

        
        assert not hasattr(fix_regularizer, '_model')


        new_model = init_model_from_family(
            family=KnownModel.ARTM,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            specific_topic_names=not_good_topic_names,
            model_params={
                'decorrelation_tau': 0.01,
                'smooth_bcg_tau': 0.05,
                'sparse_sp_tau': -0.05,
            }
        )
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
            decorr_bad_regularizer.name: decorr_bad_regularizer,
            decorr_good_regularizer.name: decorr_good_regularizer,
        }

        new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
        
        for reg in new_model.regularizers.data:
            print(f"{reg}: {new_model.regularizers[reg].tau}")

        for reg_name, reg in custom_regularizers.items():
            print(f"{reg_name}: {reg.tau}")

        
        results[key].append(new_result)

10
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
decorrelation: 0.01
None
5 2 15


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c0c0ba910>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f9c0c0baa90>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f9c0c0baac0>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.23723118212850308
sparse_theta_sp: -1.337108537108537
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10
ext_decorr_good: 10
1
No spec topics
No spec topics


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c580a46a0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f9c580a4940>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f9c0e54a9a0>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.22240423324547168
sparse_theta_sp: -1.2535392535392536
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10000000.0
ext_decorr_good: 10000000.0
2
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
decorrelation: 0.01
None
7 4 13


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c59db0d60>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f9c59db0f70>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f9c59db0df0>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.2737282870713498
sparse_theta_sp: -1.5428175428175426
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10000000.0
ext_decorr_good: 10000000.0


In [171]:
for k, r in prev_results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

10 3335.791748046875
100 3335.791748046875
1000 3335.791748046875
10000 3335.791748046875
100000.0 3335.7916666666665
1000000.0 3335.7915852864585
10000000.0 3335.791748046875


In [172]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

10 3391.75634765625
100 3391.7559407552085
1000 3391.7406412760415
10000 3391.6324869791665
100000.0 3390.8999837239585
1000000.0 3422.57373046875
10000000.0 3767.7193196614585


In [173]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    prev_r = prev_results[k]
    prev_mean_ppl = sum(v['scores']['perplexity'] for v in prev_r) / len(prev_r)

    print(k, mean_ppl - prev_mean_ppl)

10 55.964599609375
100 55.964192708333485
1000 55.948893229166515
10000 55.840738932291515
100000.0 55.10831705729197
1000000.0 86.78214518229152
10000000.0 431.9275716145835


In [148]:
# Old
#  Best: 100000.0 -14.403157552083485
# Close: 10000 -13.900065104166515

In [ ]:
# New
#  Best: 100000.0 55.10831705729197
# Close: 10000 55.840738932291515

In [174]:
prev_results = dict()
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2
DECORRELATION_TAUS = [10, 100, 1000, 10000, 1e5, 1e6, 1e7, 1e8, 1e9, 1e10]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    prev_results[key] = []
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.ARTM,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={
                'decorrelation_tau': 0.01,  # best values
                'smooth_bcg_tau': 0.05,
                'sparse_sp_tau': -0.05,
            }
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        result = fit_and_compute_scores(model, dataset)
        prev_results[key].append(result)

        
        good_topic_indices = [
            t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
        ]
        bad_topic_indices = [
            t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
        ]
        not_good_topic_indices = [
            t for t in TOPIC_INDICES if t not in good_topic_indices
        ]
        
        phi = model.get_phi()
        good_topic_names = [phi.columns[t] for t in good_topic_indices]
        bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
        not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]


        assert len(good_topic_names) > 0
        assert len(bad_topic_names) > 0

        print(len(good_topic_names), len(bad_topic_names), len(not_good_topic_names))


        fix_regularizer = FastFixPhiRegularizer(
            name='fix',
            parent_model=model._model,
            topic_names=good_topic_names,
        )
        
        bad_phi = model._model.get_phi()[bad_topic_names]
        bad_phi = deepcopy(bad_phi)
        decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
            name='ext_decorr_bad', tau=decorrelation_tau,
            topic_names=not_good_topic_names,
            other_phi=bad_phi
        )
        
        good_phi = model._model.get_phi()[good_topic_names]
        good_phi = deepcopy(good_phi)
        decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
            name='ext_decorr_good', tau=decorrelation_tau,
            topic_names=not_good_topic_names,
            other_phi=good_phi
        )

        
        assert not hasattr(fix_regularizer, '_model')


        new_model = init_model_from_family(
            family=KnownModel.ARTM,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            specific_topic_names=not_good_topic_names,
            model_params={
                'decorrelation_tau': 0.01,
                'smooth_bcg_tau': 0.05,
                'sparse_sp_tau': -0.05,
            }
        )
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
            decorr_bad_regularizer.name: decorr_bad_regularizer,
            decorr_good_regularizer.name: decorr_good_regularizer,
        }

        new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
        
        for reg in new_model.regularizers.data:
            print(f"{reg}: {new_model.regularizers[reg].tau}")

        for reg_name, reg in custom_regularizers.items():
            print(f"{reg_name}: {reg.tau}")

        
        results[key].append(new_result)

10
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
decorrelation: 0.01
None
5 2 15


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9ca8460370>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f9c3430d850>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f9ca8460400>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.23723118212850308
sparse_theta_sp: -1.337108537108537
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10
ext_decorr_good: 10
1
No spec topics
No spec topics


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c58c4ba90>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f9c58c4bd90>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f9c58c4b430>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.22240423324547168
sparse_theta_sp: -1.2535392535392536
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10000000000.0
ext_decorr_good: 10000000000.0
2
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
decorrelation: 0.01
None
7 4 13


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c59c74ee0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f9c59c749d0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f9c59c746a0>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.2737282870713498
sparse_theta_sp: -1.5428175428175426
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10000000000.0
ext_decorr_good: 10000000000.0


In [175]:
for k, r in prev_results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

10 3335.7916666666665
100 3335.7916666666665
1000 3335.791748046875
10000 3335.791748046875
100000.0 3335.7918294270835
1000000.0 3335.7916666666665
10000000.0 3335.791748046875
100000000.0 3335.7916666666665
1000000000.0 3335.7916666666665
10000000000.0 3335.7916666666665


In [176]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

10 3391.7565104166665
100 3391.7565104166665
1000 3391.7564290364585
10000 3391.75634765625
100000.0 3391.7562662760415
1000000.0 3391.7537434895835
10000000.0 3391.68212890625
100000000.0 3391.4226888020835
1000000000.0 3398.2686360677085
10000000000.0 3543.0420735677085


In [177]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    prev_r = prev_results[k]
    prev_mean_ppl = sum(v['scores']['perplexity'] for v in prev_r) / len(prev_r)

    print(k, mean_ppl - prev_mean_ppl)

10 55.96484375
100 55.96484375
1000 55.964680989583485
10000 55.964599609375
100000.0 55.96443684895803
1000000.0 55.96207682291697
10000000.0 55.890380859375
100000000.0 55.63102213541697
1000000000.0 62.47696940104197
10000000000.0 207.25040690104197


In [148]:
# Old
#  Best: 100000000.0 -14.224039713541515
# Close: 10000000.0 -13.907633463541515

In [ ]:
# New
#  Best: 100000000.0 55.63102213541697
# Close: 10000000.0 55.890380859375

In [126]:
MAX_NUM_TRAINS

20

In [184]:
results

{10000: [{'scores': {'perplexity': 3311.741943359375,
    'coherence_20': array([0.89405711]),
    'diversity_euclidean': 0.06511554726097335,
    'diversity_jensenshannon': 0.6828348172693925,
    'diversity_hellinger': 0.8022388971387356,
    'diversity_cosine': 0.8240433040000179},
   'topic_coherences': {0: 0.4683490469131406,
    1: 1.2708852483030688,
    2: 1.0016320869064537,
    3: 0.5489794585606455,
    4: 0.575541686214424,
    5: 1.0907913923515902,
    6: 0.5928311176699791,
    7: 0.7821174205147117,
    8: 1.3288948980821043,
    9: 0.9276881857861479,
    10: 0.739185861947263,
    11: 1.363958438096913,
    12: 0.82995237189913,
    13: 1.2447608642582717,
    14: 0.9497765507699979,
    15: 0.8169515987218213,
    16: 0.9020220438114654,
    17: 0.8810183824929729,
    18: 0.48978685531454025,
    19: 1.0760187567561756},
   'num_topics': {'good': 9, 'bad': 3, 'not_good': 11, 'total_bad': 3}},
  {'scores': {'perplexity': 3809.01513671875,
    'coherence_20': array([0

In [182]:
new_result

{'scores': {'perplexity': 3809.01513671875,
  'coherence_20': array([0.5945404]),
  'diversity_euclidean': 0.03684976103435768,
  'diversity_jensenshannon': 0.6252755749054375,
  'diversity_hellinger': 0.7160773729924527,
  'diversity_cosine': 0.6989911197436742},
 'topic_coherences': {0: 0.8812987258721193,
  1: 0.5314825028526059,
  2: 0.47524447874732434,
  3: 0.5514550467376536,
  4: 0.5333478286281566,
  5: 0.28828015119226597,
  6: 0.5708263826199348,
  7: 0.6065713486559028,
  8: 0.5049597466635029,
  9: 0.448355608426855,
  10: 0.6707232618359885,
  11: 0.5510471906816278,
  12: 0.4813025949440529,
  13: 0.3674447757863572,
  14: 0.5053840857065409,
  15: 0.8870700478250597,
  16: 0.9118732224903754,
  17: 0.7610267553501462,
  18: 0.8647487828879562,
  19: 0.4983655564136739}}

In [186]:
fix_regularizer._topic_names

['topic_1',
 'topic_2',
 'topic_5',
 'topic_8',
 'topic_9',
 'topic_11',
 'topic_13',
 'topic_14',
 'topic_19']

In [24]:
del model

In [32]:
result.keys()

dict_keys(['scores', 'topic_coherences', 'topic_coherences_toplen_ptw'])

In [43]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelateWithOtherPhiRegularizer

#  Best: 100000.0 55.10831705729197
# Close: 10000 55.840738932291515

DECORRELATION_TAUS = [100000]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_TOPICS:
        print(seed)

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:

            fix_regularizer = FastFixPhiRegularizer(
                name='fix',
                parent_model=prev_model._model,
                topic_names=good_topic_names,
                # tau=10 ** 12,  # TODO: had to increase tau (some topics were not saved)
            )
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items()
                if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items()
                if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # TODO: Checking that old good are at least not bad
            assert not any(t in new_bad_topic_names for t in good_topic_names)
            # TODO: ...and also that topics are preserved (maybe not "good quality", but only topics themselves)
            assert np.allclose(
                prev_model.get_phi()[good_topic_names].to_numpy(),
                phi[good_topic_names].to_numpy(),
                atol=1e-5
            )

            good_fair = len(new_good_topic_names)

            if not (set(good_topic_names) <= set(new_good_topic_names)):
                assert any(t in new_not_good_topic_names for t in good_topic_names)
                assert not any(t in new_bad_topic_names for t in good_topic_names)

                print(
                    f'DOWNFALL: some old good topics {good_topic_names}'
                    f' are not in new good topics {new_good_topic_names}.'
                    f' Manually marking them as good.'
                )

                # new_good_topic_names = list(
                #     set(new_good_topic_names).union(set(good_topic_names))
                # )
                new_expected_good_len = len(set(new_good_topic_names).union(set(good_topic_names)))

                new_good_topic_names = [
                    t for t in phi.columns
                    if t in new_good_topic_names or t in good_topic_names
                ]

                assert len(new_good_topic_names) == new_expected_good_len

                # new_not_good_topic_names = list(
                #     set(new_not_good_topic_names).difference(set(good_topic_names))
                # )
                new_expected_not_good_len = len(set(new_not_good_topic_names).difference(set(good_topic_names)))
                
                new_not_good_topic_names = [
                    t for t in phi.columns
                    if t in new_not_good_topic_names and t not in good_topic_names
                ]

                assert len(new_not_good_topic_names) == new_expected_not_good_len

                good_topic_indices = [phi.columns.get_loc(t) for t in new_good_topic_names]
                not_good_topic_indices = [phi.columns.get_loc(t) for t in new_not_good_topic_names]
            
            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # TODO: remove this assert because "intra-goodness" depends on topic model
            # assert set(good_topic_names) <= set(new_good_topic_names), f'{good_topic_names} -- {new_good_topic_names}'

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': good_fair,
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

100000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.24148051194680664
sparse_theta_sp: -1.3690669651493796
decorrelation: 0.01
None
num_topics: {'good': 3, 'good_fair': 3, 'bad': 2, 'not_good': 17, 'total_bad': 2}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f064e6db940>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f06c36ab130>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f04d32380d0>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.2840947199374196
sparse_theta_sp: -1.6106670178227993
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 5, 'good_fair': 5, 'bad': 4, 'not_good': 15, 'total_bad': 6}
2


...

{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f04bfedd220>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f0585c1d250>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f06191d5ee0>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.3715084799181641
sparse_theta_sp: -2.1062568694605837
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
DOWNFALL: some old good topics ['topic_0', 'topic_4', 'topic_5', 'topic_10', 'topic_15', 'topic_16', 'topic_19'] are not in new good topics ['topic_0', 'topic_5', 'topic_15', 'topic_16', 'topic_19']. Manually marking them as good.
num_topics: {'good': 7, 'good_fair': 5, 'bad': 7, 'not_good': 13, 'total_bad': 107}
18


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f05daa48430>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f06c5f87a30>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f04bfedd130>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.3715084799181641
sparse_theta_sp: -2.1062568694605837
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
DOWNFALL: some old good topics ['topic_0', 'topic_4', 'topic_5', 'topic_10', 'topic_15', 'topic_16', 'topic_19'] are not in new good topics ['topic_0', 'topic_5', 'topic_16', 'topic_19']. Manually marking them as good.
SUCCESS: less bad topics
num_topics: {'good': 7, 'good_fair': 4, 'bad': 6, 'not_good': 13, 'total_bad': 113}
19


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f05f0a0d4c0>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f06192c4640>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f04c3200cd0>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.3715084799181641
sparse_theta_sp: -2.1062568694605837
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
DOWNFALL: some old good topics ['topic_0', 'topic_4', 'topic_5', 'topic_10', 'topic_15', 'topic_16', 'topic_19'] are not in new good topics ['topic_0', 'topic_5', 'topic_16', 'topic_19']. Manually marking them as good.
DOWNFALL: more bad topics...
num_topics: {'good': 7, 'good_fair': 4, 'bad': 7, 'not_good': 13, 'total_bad': 120}


In [45]:
results.keys()

dict_keys([100000])

In [46]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [47]:
import json

SAVE_FOLDER = 'results_intra/postnauka'

! mkdir -p $SAVE_FOLDER

In [48]:
for k, r in results.items():
    with open(SAVE_FOLDER + f'/iterative_{int(k)}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [49]:
! ls $SAVE_FOLDER

decorrelation_with_cohs.json  lda_with_cohs.json   sparse_with_cohs.json
iterative_100000.json	      plsa_with_cohs.json  tless_with_cohs.json


In [56]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelateWithOtherPhiRegularizer

#  Best: 100000.0 55.10831705729197
# Close: 10000 55.840738932291515

DECORRELATION_TAUS = [100000]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_TOPICS:
        print(seed)

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:

            fix_regularizer = FastFixPhiRegularizer(
                name='fix',
                parent_model=prev_model._model,
                topic_names=good_topic_names,
                # tau=10 ** 12,  # TODO: had to increase tau (some topics were not saved)
            )
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                # decorr_good_regularizer.name: decorr_good_regularizer,
            }
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items()
                if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items()
                if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # TODO: Checking that old good are at least not bad
            assert not any(t in new_bad_topic_names for t in good_topic_names)
            # TODO: ...and also that topics are preserved (maybe not "good quality", but only topics themselves)
            assert np.allclose(
                prev_model.get_phi()[good_topic_names].to_numpy(),
                phi[good_topic_names].to_numpy(),
                atol=1e-5
            )

            good_fair = len(new_good_topic_names)

            if not (set(good_topic_names) <= set(new_good_topic_names)):
                assert any(t in new_not_good_topic_names for t in good_topic_names)
                assert not any(t in new_bad_topic_names for t in good_topic_names)

                print(
                    f'DOWNFALL: some old good topics {good_topic_names}'
                    f' are not in new good topics {new_good_topic_names}.'
                    f' Manually marking them as good.'
                )

                # new_good_topic_names = list(
                #     set(new_good_topic_names).union(set(good_topic_names))
                # )
                new_expected_good_len = len(set(new_good_topic_names).union(set(good_topic_names)))

                new_good_topic_names = [
                    t for t in phi.columns
                    if t in new_good_topic_names or t in good_topic_names
                ]

                assert len(new_good_topic_names) == new_expected_good_len

                # new_not_good_topic_names = list(
                #     set(new_not_good_topic_names).difference(set(good_topic_names))
                # )
                new_expected_not_good_len = len(set(new_not_good_topic_names).difference(set(good_topic_names)))
                
                new_not_good_topic_names = [
                    t for t in phi.columns
                    if t in new_not_good_topic_names and t not in good_topic_names
                ]

                assert len(new_not_good_topic_names) == new_expected_not_good_len

                good_topic_indices = [phi.columns.get_loc(t) for t in new_good_topic_names]
                not_good_topic_indices = [phi.columns.get_loc(t) for t in new_not_good_topic_names]
            
            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # TODO: remove this assert because "intra-goodness" depends on topic model
            # assert set(good_topic_names) <= set(new_good_topic_names), f'{good_topic_names} -- {new_good_topic_names}'

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': good_fair,
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

100000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.24148051194680664
sparse_theta_sp: -1.3690669651493796
decorrelation: 0.01
None
num_topics: {'good': 3, 'good_fair': 3, 'bad': 2, 'not_good': 17, 'total_bad': 2}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f06192bdb50>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f04b9921e50>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.2840947199374196
sparse_theta_sp: -1.6106670178227993
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 5, 'good_fair': 5, 'bad': 5, 'not_good': 15, 'total_bad': 7}
2


...

{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f04a0600f70>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f049866c280>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.3715084799181641
sparse_theta_sp: -2.1062568694605837
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
DOWNFALL: some old good topics ['topic_0', 'topic_4', 'topic_5', 'topic_10', 'topic_15', 'topic_16', 'topic_19'] are not in new good topics ['topic_0', 'topic_5', 'topic_15', 'topic_16', 'topic_19']. Manually marking them as good.
SUCCESS: less bad topics
num_topics: {'good': 7, 'good_fair': 5, 'bad': 6, 'not_good': 13, 'total_bad': 103}
18


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f047d833760>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f05dac8bf40>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.3715084799181641
sparse_theta_sp: -2.1062568694605837
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
DOWNFALL: some old good topics ['topic_0', 'topic_4', 'topic_5', 'topic_10', 'topic_15', 'topic_16', 'topic_19'] are not in new good topics ['topic_0', 'topic_5', 'topic_19']. Manually marking them as good.
DOWNFALL: more bad topics...
num_topics: {'good': 7, 'good_fair': 3, 'bad': 8, 'not_good': 13, 'total_bad': 111}
19


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f04b1685520>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer object at 0x7f062deed280>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.3715084799181641
sparse_theta_sp: -2.1062568694605837
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
DOWNFALL: some old good topics ['topic_0', 'topic_4', 'topic_5', 'topic_10', 'topic_15', 'topic_16', 'topic_19'] are not in new good topics ['topic_0', 'topic_5', 'topic_16', 'topic_19']. Manually marking them as good.
SUCCESS: less bad topics
num_topics: {'good': 7, 'good_fair': 4, 'bad': 7, 'not_good': 13, 'total_bad': 118}


In [57]:
results.keys()

dict_keys([100000])

In [58]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [59]:
for k, r in results.items():
    with open(SAVE_FOLDER + f'/iterative_{int(k)}_no_decorr_good.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [60]:
! ls $SAVE_FOLDER

decorrelation_with_cohs.json	      lda_with_cohs.json
iterative_100000.json		      plsa_with_cohs.json
iterative_100000_no_decorr_good.json  sparse_with_cohs.json
iterative2_100000000.json	      tless_with_cohs.json


In [211]:
results.keys()

dict_keys([10000, 100000.0])

In [212]:
results[10000][-1]

{'scores': {'perplexity': 3652.7041015625,
  'coherence_20': 0.9516712577049755,
  'diversity_euclidean': 0.0836073724255812,
  'diversity_jensenshannon': 0.7118096839804591,
  'diversity_hellinger': 0.8378503989777749,
  'diversity_cosine': 0.8669797597740407},
 'topic_coherences': {0: 0.9157857043839753,
  1: 1.0473690743951338,
  2: 1.302009849349046,
  3: 1.1402745197980073,
  4: 1.0884573939452766,
  5: 1.0168754020127426,
  6: 0.7151896319173402,
  7: 0.603791673467527,
  8: 0.5736267875313644,
  9: 0.9496766038826497,
  10: 1.1562135471080357,
  11: 0.49139231251357834,
  12: 0.9864582817408862,
  13: 0.7660369894434516,
  14: 1.0013359645176345,
  15: 0.938758599734583,
  16: 1.2898827651571152,
  17: 0.997072793474575,
  18: 0.9516392681906888,
  19: 1.1015779915359032},
 'num_topics': {'good': 15, 'bad': 1, 'not_good': 5, 'total_bad': 39}}

In [213]:
results[100000.0][-1]

{'scores': {'perplexity': 3791.001220703125,
  'coherence_20': 1.0257106864524834,
  'diversity_euclidean': 0.19351423230474737,
  'diversity_jensenshannon': inf,
  'diversity_hellinger': 0.8766665588874561,
  'diversity_cosine': nan},
 'topic_coherences': {0: 0.6354215003003565,
  1: 0.6151009034375062,
  2: 1.302009849349046,
  3: 0.9214287215829502,
  4: 1.0884573939452766,
  5: 1.0168754020127426,
  6: 1.14367941540882,
  7: 0.9419151443954846,
  8: 0.9694617363799289,
  9: 1.1009542380952984,
  10: 1.1562135471080357,
  11: 0.9765581860494639,
  12: 1.1521117689445122,
  13: 0.9412741875348247,
  14: 0.9925343464340454,
  15: 1.020407105804347,
  16: 1.2898827651571152,
  17: 1.0524228260033137,
  18: 1.0296176298910718,
  19: 1.1678870612155294},
 'num_topics': {'good': 18, 'bad': 0, 'not_good': 2, 'total_bad': 21}}

In [32]:
results[100000.0][-1]

{'scores': {'perplexity': 3789.63671875,
  'coherence_20': 1.0192302655028405,
  'diversity_euclidean': 0.27780393576980444,
  'diversity_jensenshannon': 0.7630736309319524,
  'diversity_hellinger': 0.9058558181488267,
  'diversity_cosine': 0.929081787382234},
 'topic_coherences': {0: 0.5454667436822921,
  1: 0.5754472410627138,
  2: 1.302009849349046,
  3: 0.9214287215829502,
  4: 1.0884573939452766,
  5: 1.0168754020127426,
  6: 1.14367941540882,
  7: 0.9419151443954846,
  8: 0.9694617363799289,
  9: 1.1009542380952984,
  10: 1.1562135471080357,
  11: 0.9765581860494639,
  12: 1.1521117689445122,
  13: 0.9412741875348247,
  14: 0.9925343464340454,
  15: 1.020407105804347,
  16: 1.2898827651571152,
  17: 1.0524228260033137,
  18: 1.0296176298910718,
  19: 1.1678870612155294},
 'num_topics': {'good': 18, 'bad': 1, 'not_good': 2, 'total_bad': 20}}

In [33]:
phi = prev_model.get_phi()

target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
target_topic_names = [phi.columns[i] for i in target_topic_indices]
diversity_scores = [
    DiversityScore(
        name=f'diversity_{metric}',
        metric=metric,
        topic_names=target_topic_names,
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

for s in diversity_scores:
    print(f'{s.name}: {s.call(prev_model)}')

diversity_euclidean: 0.27780393576980444
diversity_jensenshannon: 0.7630736309319524
diversity_hellinger: 0.9058558181488267
diversity_cosine: 0.929081787382234


In [34]:
view_model(prev_model, dataset)

topic_0 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 вулкан 
 1.000000 
 
 
 акт 
 0.000000 
 
 
 край 
 0.000000 
 
 
 медиа 
 0.000000 
 
 
 отличный 
 0.000000

{}

topic_1 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 кавказ 
 0.935930 
 
 
 берег 
 0.064070 
 
 
 архангел 
 0.000000 
 
 
 край 
 0.000000 
 
 
 раскрыть 
 0.000000

{}

topic_2 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 caption 
 0.025620 
 
 
 align 
 0.012790 
 
 
 width 
 0.012790 
 
 
 attachment 
 0.012610 
 
 
 свет 
 0.012440

{'322.txt': 0.6950722,
 '1281.txt': 0.65286434,
 '2564.txt': 0.6389526,
 '1823.txt': 0.6110392,
 '2571.txt': 0.6082495,
 '1822.txt': 0.607679,
 '3325.txt': 0.6046282,
 '17.txt': 0.5958985,
 '2222.txt': 0.5856556,
 '3122.txt': 0.57845175}

topic_3 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 война 
 0.016210 
 
 
 россия 
 0.015520 
 
 
 русский 
 0.015170 
 
 
 революция 
 0.010160 
 
 
 советский 
 0.008690

{'601.txt': 0.41019967,
 '292.txt': 0.3975281,
 '200.txt': 0.31401595,
 '1261.txt': 0.2879542,
 '3352.txt': 0.27771163}

topic_4 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 право 
 0.019670 
 
 
 закон 
 0.009850 
 
 
 сталин 
 0.008530 
 
 
 власть 
 0.007170 
 
 
 история 
 0.006090

{'1862.txt': 0.81326944,
 '2390.txt': 0.69851804,
 '3387.txt': 0.65967566,
 '672.txt': 0.64996207,
 '775.txt': 0.61321616,
 '1131.txt': 0.60506326,
 '1767.txt': 0.6045837,
 '2975.txt': 0.5961673,
 '3176.txt': 0.59539086,
 '3088.txt': 0.57747215}

In [225]:
sum(phi['topic_1'])

0.0

In [35]:
top = 20
coherence_score = TopTokenCoherence(
    name=f'coherence_{top}',
    func=create_pmi_top_function(
        occurences, co_occurences,
        dataset.get_dataset().shape[0], [top],
        topic_indices=target_topic_indices,
        co_occurrences_smooth=1e-2,
    )
)

In [36]:
coherence_score.call(prev_model)

array([1.01923027])

In [37]:
coherence_score.call_by_topic(prev_model)

{0: array([0.54546674]),
 1: array([0.57544724]),
 2: array([1.30200985]),
 3: array([0.92142872]),
 4: array([1.08845739]),
 5: array([1.0168754]),
 6: array([1.14367942]),
 7: array([0.94191514]),
 8: array([0.96946174]),
 9: array([1.10095424]),
 10: array([1.15621355]),
 11: array([0.97655819]),
 12: array([1.15211177]),
 13: array([0.94127419]),
 14: array([0.99253435]),
 15: array([1.02040711]),
 16: array([1.28988277]),
 17: array([1.05242283]),
 18: array([1.02961763]),
 19: array([1.16788706])}

In [246]:
from scipy.spatial.distance import pdist

condensed_distances = pdist(phi.T, metric='cosine')

In [247]:
condensed_distances

array([       nan, 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 0.94932407, 1.        , 0.9996893 , 0.99947131,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan, 0.95226152,
       0.91858436, 0.84067465, 0.98570107, 0.95536781, 0.93841504,
       0.91195556, 0.81952628, 0.99693492, 0.99000703, 0.90857287,
       0.897325  , 0.89665542, 0.88727302, 0.91061367, 0.9345907 ,
       0.94975521, 0.77415403, 0.83667812, 0.92408579, 0.96343722,
       0.93270342, 0.9717384 , 0.81929878, 0.93403098, 0.9877177 ,
       0.98828042, 0.82303836, 0.90675104, 0.73582458, 0.9334561 ,
       0.78237281, 0.8943664 , 0.91692259, 0.80166188, 0.88287

In [244]:
# condensed_distances[~np.isfinite(condensed_distances)] = 0

In [248]:
condensed_distances[np.isfinite(condensed_distances)]

array([1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       0.94932407, 1.        , 0.9996893 , 0.99947131, 0.95226152,
       0.91858436, 0.84067465, 0.98570107, 0.95536781, 0.93841504,
       0.91195556, 0.81952628, 0.99693492, 0.99000703, 0.90857287,
       0.897325  , 0.89665542, 0.88727302, 0.91061367, 0.9345907 ,
       0.94975521, 0.77415403, 0.83667812, 0.92408579, 0.96343722,
       0.93270342, 0.9717384 , 0.81929878, 0.93403098, 0.9877177 ,
       0.98828042, 0.82303836, 0.90675104, 0.73582458, 0.9334561 ,
       0.78237281, 0.8943664 , 0.91692259, 0.80166188, 0.88287796,
       0.99319125, 0.80747649, 0.90576928, 0.80097294, 0.86966945,
       0.99220407, 0.97159118, 0.79524658, 0.85588285, 0.71155101,
       0.88010361, 0.76435427, 0.92463188, 0.90484961, 0.67535313,
       0.98889709, 0.9472439 , 0.86250206, 0.8786938 , 0.69777

In [252]:
condensed_distances = pdist(phi.T, metric='jensenshannon')

In [253]:
len(condensed_distances)

210

In [254]:
condensed_distances[np.isfinite(condensed_distances)]

array([0.83255461, 0.83255461, 0.83255461, 0.83255461, 0.83255461,
       0.83255461, 0.83255461, 0.83255461, 0.83255461, 0.83255461,
       0.83255461, 0.83255461, 0.83255461, 0.83255461, 0.83255461,
       0.82802436, 0.83255461, 0.83245481, 0.83251028, 0.765152  ,
       0.72735422, 0.65473411, 0.80893922, 0.77133901, 0.76819008,
       0.72685403, 0.64583768, 0.82509744, 0.82683813, 0.74428594,
       0.71080644, 0.71440577, 0.68847481, 0.72293876, 0.73349144,
       0.71004757, 0.65152476, 0.70896455, 0.74516122, 0.80661507,
       0.76236171, 0.79451686, 0.70221644, 0.7452968 , 0.8215549 ,
       0.82409651, 0.71841013, 0.73682448, 0.6945145 , 0.74529442,
       0.69433464, 0.76630807, 0.72245097, 0.70305571, 0.70510286,
       0.81433465, 0.68533599, 0.7486963 , 0.66462088, 0.70240846,
       0.81993557, 0.81780973, 0.67868887, 0.6848692 , 0.6305586 ,
       0.69941677, 0.662727  , 0.73583013, 0.68252334, 0.61874214,
       0.80882547, 0.75610061, 0.75167062, 0.70565034, 0.57763

In [256]:
len(condensed_distances[np.isfinite(condensed_distances)])

190

In [264]:
! cat results/postnauka/iterative_100000.json | grep NaN

            "diversity_cosine": NaN
            "diversity_cosine": NaN
            "diversity_cosine": NaN
            "diversity_cosine": NaN
            "diversity_cosine": NaN
            "diversity_cosine": NaN
            "diversity_cosine": NaN


In [265]:
topnum.__file__

'/home/alekseev_v/projects/iterative/../OptimalNumberOfTopics/topnum/__init__.py'

In [50]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelateWithOtherPhiRegularizer2

DECORRELATION_TAUS = [100000000]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_TOPICS:
        print(seed)

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:

            fix_regularizer = FastFixPhiRegularizer(
                name='fix',
                parent_model=prev_model._model,
                topic_names=good_topic_names,
                # tau=10 ** 12,  # TODO: had to increase tau (some topics were not saved)
            )
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items()
                if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items()
                if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # TODO: Checking that old good are at least not bad
            assert not any(t in new_bad_topic_names for t in good_topic_names)
            # TODO: ...and also that topics are preserved (maybe not "good quality", but only topics themselves)
            assert np.allclose(
                prev_model.get_phi()[good_topic_names].to_numpy(),
                phi[good_topic_names].to_numpy(),
                atol=1e-5
            )

            good_fair = len(new_good_topic_names)

            if not (set(good_topic_names) <= set(new_good_topic_names)):
                assert any(t in new_not_good_topic_names for t in good_topic_names)
                assert not any(t in new_bad_topic_names for t in good_topic_names)

                print(
                    f'DOWNFALL: some old good topics {good_topic_names}'
                    f' are not in new good topics {new_good_topic_names}.'
                    f' Manually marking them as good.'
                )

                # new_good_topic_names = list(
                #     set(new_good_topic_names).union(set(good_topic_names))
                # )
                new_expected_good_len = len(set(new_good_topic_names).union(set(good_topic_names)))

                new_good_topic_names = [
                    t for t in phi.columns
                    if t in new_good_topic_names or t in good_topic_names
                ]

                assert len(new_good_topic_names) == new_expected_good_len

                # new_not_good_topic_names = list(
                #     set(new_not_good_topic_names).difference(set(good_topic_names))
                # )
                new_expected_not_good_len = len(set(new_not_good_topic_names).difference(set(good_topic_names)))
                
                new_not_good_topic_names = [
                    t for t in phi.columns
                    if t in new_not_good_topic_names and t not in good_topic_names
                ]

                assert len(new_not_good_topic_names) == new_expected_not_good_len

                good_topic_indices = [phi.columns.get_loc(t) for t in new_good_topic_names]
                not_good_topic_indices = [phi.columns.get_loc(t) for t in new_not_good_topic_names]
            
            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # TODO: remove this assert because "intra-goodness" depends on topic model
            # assert set(good_topic_names) <= set(new_good_topic_names), f'{good_topic_names} -- {new_good_topic_names}'

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': good_fair,
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

100000000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.24148051194680664
sparse_theta_sp: -1.3690669651493796
decorrelation: 0.01
None
num_topics: {'good': 3, 'good_fair': 3, 'bad': 2, 'not_good': 17, 'total_bad': 2}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f062deedc10>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f06192c5d60>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f04a9d7b190>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.2840947199374196
sparse_theta_sp: -1.6106670178227993
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 5, 'good_fair': 5, 'bad': 4, 'not_good': 15, 'total_bad': 6}
2


...

{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f06e4ffdc40>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f06e446e9a0>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f049c5638e0>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.3715084799181641
sparse_theta_sp: -2.1062568694605837
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: some old good topics ['topic_0', 'topic_4', 'topic_5', 'topic_10', 'topic_15', 'topic_16', 'topic_19'] are not in new good topics ['topic_0', 'topic_5', 'topic_10', 'topic_15', 'topic_16', 'topic_19']. Manually marking them as good.
SUCCESS: less bad topics
num_topics: {'good': 7, 'good_fair': 6, 'bad': 6, 'not_good': 13, 'total_bad': 109}
18


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f04d36cc790>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f06537eca00>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f04c31b8460>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.3715084799181641
sparse_theta_sp: -2.1062568694605837
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: some old good topics ['topic_0', 'topic_4', 'topic_5', 'topic_10', 'topic_15', 'topic_16', 'topic_19'] are not in new good topics ['topic_0', 'topic_5', 'topic_16', 'topic_19']. Manually marking them as good.
DOWNFALL: more bad topics...
num_topics: {'good': 7, 'good_fair': 4, 'bad': 7, 'not_good': 13, 'total_bad': 116}
19


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f046801aa00>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f067a551160>, 'ext_decorr_good': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f04bd447ca0>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.3715084799181641
sparse_theta_sp: -2.1062568694605837
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: some old good topics ['topic_0', 'topic_4', 'topic_5', 'topic_10', 'topic_15', 'topic_16', 'topic_19'] are not in new good topics ['topic_0', 'topic_4', 'topic_5', 'topic_16', 'topic_19']. Manually marking them as good.
num_topics: {'good': 7, 'good_fair': 5, 'bad': 7, 'not_good': 13, 'total_bad': 123}


In [40]:
1

1

In [51]:
results.keys()

dict_keys([100000000])

In [52]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [53]:
for k, r in results.items():
    with open(SAVE_FOLDER + f'/iterative2_{int(k)}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [54]:
! ls $SAVE_FOLDER

decorrelation_with_cohs.json  lda_with_cohs.json     tless_with_cohs.json
iterative_100000.json	      plsa_with_cohs.json
iterative2_100000000.json     sparse_with_cohs.json


In [61]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelateWithOtherPhiRegularizer2

DECORRELATION_TAUS = [100000000]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_TOPICS:
        print(seed)

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences_toplen_ptw'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:

            fix_regularizer = FastFixPhiRegularizer(
                name='fix',
                parent_model=prev_model._model,
                topic_names=good_topic_names,
                # tau=10 ** 12,  # TODO: had to increase tau (some topics were not saved)
            )
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                # decorr_good_regularizer.name: decorr_good_regularizer,
            }
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items()
                if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences_toplen_ptw'].items()
                if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # TODO: Checking that old good are at least not bad
            assert not any(t in new_bad_topic_names for t in good_topic_names)
            # TODO: ...and also that topics are preserved (maybe not "good quality", but only topics themselves)
            assert np.allclose(
                prev_model.get_phi()[good_topic_names].to_numpy(),
                phi[good_topic_names].to_numpy(),
                atol=1e-5
            )

            good_fair = len(new_good_topic_names)

            if not (set(good_topic_names) <= set(new_good_topic_names)):
                assert any(t in new_not_good_topic_names for t in good_topic_names)
                assert not any(t in new_bad_topic_names for t in good_topic_names)

                print(
                    f'DOWNFALL: some old good topics {good_topic_names}'
                    f' are not in new good topics {new_good_topic_names}.'
                    f' Manually marking them as good.'
                )

                # new_good_topic_names = list(
                #     set(new_good_topic_names).union(set(good_topic_names))
                # )
                new_expected_good_len = len(set(new_good_topic_names).union(set(good_topic_names)))

                new_good_topic_names = [
                    t for t in phi.columns
                    if t in new_good_topic_names or t in good_topic_names
                ]

                assert len(new_good_topic_names) == new_expected_good_len

                # new_not_good_topic_names = list(
                #     set(new_not_good_topic_names).difference(set(good_topic_names))
                # )
                new_expected_not_good_len = len(set(new_not_good_topic_names).difference(set(good_topic_names)))
                
                new_not_good_topic_names = [
                    t for t in phi.columns
                    if t in new_not_good_topic_names and t not in good_topic_names
                ]

                assert len(new_not_good_topic_names) == new_expected_not_good_len

                good_topic_indices = [phi.columns.get_loc(t) for t in new_good_topic_names]
                not_good_topic_indices = [phi.columns.get_loc(t) for t in new_not_good_topic_names]
            
            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # TODO: remove this assert because "intra-goodness" depends on topic model
            # assert set(good_topic_names) <= set(new_good_topic_names), f'{good_topic_names} -- {new_good_topic_names}'

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'good_fair': good_fair,
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }
            results[key][-1]['good_topic_indices'] = good_topic_indices

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

100000000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.24148051194680664
sparse_theta_sp: -1.3690669651493796
decorrelation: 0.01
None
num_topics: {'good': 3, 'good_fair': 3, 'bad': 2, 'not_good': 17, 'total_bad': 2}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f04b2374730>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f04ad82c790>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.2840947199374196
sparse_theta_sp: -1.6106670178227993
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 5, 'good_fair': 5, 'bad': 4, 'not_good': 15, 'total_bad': 6}
2


...

{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f049483cf70>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f044881df40>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.34497215992400954
sparse_theta_sp: -1.9558099502133992
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
num_topics: {'good': 6, 'good_fair': 6, 'bad': 7, 'not_good': 14, 'total_bad': 108}
18


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f04d36cc790>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f064e6baa60>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.34497215992400954
sparse_theta_sp: -1.9558099502133992
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
DOWNFALL: some old good topics ['topic_0', 'topic_4', 'topic_5', 'topic_15', 'topic_16', 'topic_19'] are not in new good topics ['topic_0', 'topic_5', 'topic_16', 'topic_19']. Manually marking them as good.
DOWNFALL: more bad topics...
num_topics: {'good': 6, 'good_fair': 4, 'bad': 8, 'not_good': 14, 'total_bad': 116}
19


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <topnum.regularizers.fix_phi.FastFixPhiRegularizer object at 0x7f040cc54460>, 'ext_decorr_bad': <topnum.regularizers.decorrelate_with_other_phi.DecorrelateWithOtherPhiRegularizer2 object at 0x7f0499574100>}
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.34497215992400954
sparse_theta_sp: -1.9558099502133992
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
DOWNFALL: some old good topics ['topic_0', 'topic_4', 'topic_5', 'topic_15', 'topic_16', 'topic_19'] are not in new good topics ['topic_0', 'topic_4', 'topic_5', 'topic_16', 'topic_19']. Manually marking them as good.
SUCCESS: less bad topics
num_topics: {'good': 6, 'good_fair': 5, 'bad': 7, 'not_good': 14, 'total_bad': 123}


In [62]:
results.keys()

dict_keys([100000000])

In [63]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [64]:
for k, r in results.items():
    with open(SAVE_FOLDER + f'/iterative2_{int(k)}_no_decorr_good.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [65]:
! ls $SAVE_FOLDER

decorrelation_with_cohs.json		  lda_with_cohs.json
iterative_100000.json			  plsa_with_cohs.json
iterative_100000_no_decorr_good.json	  sparse_with_cohs.json
iterative2_100000000.json		  tless_with_cohs.json
iterative2_100000000_no_decorr_good.json


In [46]:
results[10000000][-1]

{'scores': {'perplexity': 3597.001708984375,
  'coherence_20': 0.9149112254468618,
  'diversity_euclidean': 0.0731302527819504,
  'diversity_jensenshannon': 0.6986256151016943,
  'diversity_hellinger': 0.8211371346438057,
  'diversity_cosine': 0.84910814424976},
 'topic_coherences': {0: 0.6665189780504528,
  1: 0.7772552016342738,
  2: 1.302009849349046,
  3: 0.6911167892368315,
  4: 1.0884573939452766,
  5: 1.0168754020127426,
  6: 0.525434759315788,
  7: 0.5894657808588326,
  8: 0.5339315418590312,
  9: 0.9496766038826499,
  10: 1.1562135471080357,
  11: 1.0361673256302522,
  12: 0.9864582817408862,
  13: 0.7166985655165372,
  14: 0.9830130707037311,
  15: 0.938758599734583,
  16: 1.2898827651571152,
  17: 0.997072793474575,
  18: 0.9516392681906888,
  19: 1.1015779915359032},
 'num_topics': {'good': 13, 'bad': 2, 'not_good': 7, 'total_bad': 47}}

In [47]:
results[100000000][-1]

{'scores': {'perplexity': 3749.740966796875,
  'coherence_20': 1.0035540046975566,
  'diversity_euclidean': 0.17922226431951543,
  'diversity_jensenshannon': 0.7398631312137611,
  'diversity_hellinger': 0.8581046722219192,
  'diversity_cosine': 0.9074182569493797},
 'topic_coherences': {0: 0.9206962025174021,
  1: 0.5766228619158285,
  2: 1.302009849349046,
  3: 1.0097633592414297,
  4: 1.0884573939452766,
  5: 1.0168754020127426,
  6: 0.924416732773585,
  7: 0.6151009034375062,
  8: 0.9170213575321767,
  9: 0.9177214062346242,
  10: 1.1562135471080357,
  11: 0.9271215190879913,
  12: 0.9864582817408863,
  13: 1.3079228917282946,
  14: 0.9637786161029726,
  15: 0.938758599734583,
  16: 1.2898827651571152,
  17: 0.9970727934745751,
  18: 1.0472985496415317,
  19: 1.1678870612155292},
 'num_topics': {'good': 18, 'bad': 0, 'not_good': 2, 'total_bad': 18}}

In [48]:
view_model(prev_model, dataset)

topic_0 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 женщина 
 0.030190 
 
 
 ребёнок 
 0.025650 
 
 
 мужчина 
 0.018750 
 
 
 семья 
 0.013200 
 
 
 родитель 
 0.008690

{'802.txt': 0.6790531,
 '3193.txt': 0.49982226,
 '1192.txt': 0.4284458,
 '846.txt': 0.4148499,
 '3268.txt': 0.34310094,
 '3279.txt': 0.33936322,
 '2149.txt': 0.28610045,
 '2.txt': 0.25491595}

topic_1 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 толстой 
 0.969100 
 
 
 этнический 
 0.030900 
 
 
 архангел 
 0.000000 
 
 
 двоичный 
 0.000000 
 
 
 край 
 0.000000

{}

topic_2 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 caption 
 0.025620 
 
 
 align 
 0.012790 
 
 
 width 
 0.012790 
 
 
 attachment 
 0.012610 
 
 
 свет 
 0.012440

{'322.txt': 0.69511205,
 '1281.txt': 0.6529927,
 '2564.txt': 0.6409627,
 '1822.txt': 0.61511415,
 '1823.txt': 0.61214083,
 '2571.txt': 0.61020726,
 '3325.txt': 0.60370445,
 '17.txt': 0.5868808,
 '2222.txt': 0.58600223,
 '3122.txt': 0.58266467}

topic_3 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 город 
 0.036830 
 
 
 пространство 
 0.019050 
 
 
 дом 
 0.010460 
 
 
 архитектура 
 0.009240 
 
 
 городской 
 0.008860

{'227.txt': 0.28699762, '200.txt': 0.19679928}

topic_4 
 
 
 modality 
 token 
   
 
 
 
 
 @word 
 право 
 0.019670 
 
 
 закон 
 0.009850 
 
 
 сталин 
 0.008530 
 
 
 власть 
 0.007170 
 
 
 история 
 0.006090

{'1862.txt': 0.81434906,
 '2390.txt': 0.7064004,
 '672.txt': 0.67176133,
 '3387.txt': 0.6634288,
 '1767.txt': 0.6122575,
 '1131.txt': 0.606718,
 '775.txt': 0.6034629,
 '2975.txt': 0.5950686,
 '3176.txt': 0.589563,
 '3088.txt': 0.57827413}

In [49]:
! ls results/postnauka

decorrelation.json	   iterative_10000.json       lda.json	   tless.json
iterative_100000.json	   iterative2_100000000.json  plsa.json
iterative_100000_new.json  iterative2_10000000.json   sparse.json


In [51]:
! tail -n 50 results/postnauka/iterative_100000_new.json

            "17": 1.0524228260033137,
            "18": 1.0296176298910718,
            "19": 1.1678870612155294
        },
        "num_topics": {
            "good": 18,
            "bad": 0,
            "not_good": 2,
            "total_bad": 19
        }
    },
    {
        "scores": {
            "perplexity": 3789.63671875,
            "coherence_20": 1.0192302655028405,
            "diversity_euclidean": 0.27780393576980444,
            "diversity_jensenshannon": 0.7630736309319524,
            "diversity_hellinger": 0.9058558181488267,
            "diversity_cosine": 0.929081787382234
        },
        "topic_coherences": {
            "0": 0.5454667436822921,
            "1": 0.5754472410627138,
            "2": 1.302009849349046,
            "3": 0.9214287215829502,
            "4": 1.0884573939452766,
            "5": 1.0168754020127426,
            "6": 1.14367941540882,
            "7": 0.9419151443954846,
            "8": 0.9694617363799289,
            "9": 1.100954238

In [52]:
! tail -n 50 results/postnauka/iterative2_100000000.json

            "17": 0.9970727934745751,
            "18": 1.0472985496415317,
            "19": 1.1678870612155292
        },
        "num_topics": {
            "good": 18,
            "bad": 0,
            "not_good": 2,
            "total_bad": 18
        }
    },
    {
        "scores": {
            "perplexity": 3749.740966796875,
            "coherence_20": 1.0035540046975566,
            "diversity_euclidean": 0.17922226431951543,
            "diversity_jensenshannon": 0.7398631312137611,
            "diversity_hellinger": 0.8581046722219192,
            "diversity_cosine": 0.9074182569493797
        },
        "topic_coherences": {
            "0": 0.9206962025174021,
            "1": 0.5766228619158285,
            "2": 1.302009849349046,
            "3": 1.0097633592414297,
            "4": 1.0884573939452766,
            "5": 1.0168754020127426,
            "6": 0.924416732773585,
            "7": 0.6151009034375062,
            "8": 0.9170213575321767,
            "9": 0.917

## Ablation Study

In [54]:
DECORRELATION_TAU = 100000

ALL_PARAMS = [
    (0, 1, 1),
    (1, 0, 1),
    (1, 1, 0),

    (1, 0, 0),
    (0, 1, 0),
    (0, 0, 1),
]

In [56]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer

#  Best: 100000.0 55.10831705729197
# Close: 10000 55.840738932291515

for params in ALL_PARAMS:
    key = params
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_TOPICS:
        print(seed)

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:
            custom_regularizers = dict()
            
            if params[0]:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)

            if params[1]:
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

(0, 1, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
decorrelation: 0.01
None
num_topics: {'good': 5, 'bad': 2, 'not_good': 15, 'total_bad': 2}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f089c64c970>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f089c64c9d0>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.23723118212850308
sparse_theta_sp: -1.337108537108537
decorrelation: 0.01
ext_decorr_bad: 100000
ext_decorr_good: 100000
DOWNFALL: some good topics lost...
num_topics: {'good': 5, 'bad': 2, 'not_good': 15, 'total_bad': 4}
2


...

{'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f08858dd550>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.2965389776606289
sparse_theta_sp: -1.6713856713856714
decorrelation: 0.01
ext_decorr_good: 100000
DOWNFALL: some good topics lost...
SUCCESS: less bad topics
num_topics: {'good': 5, 'bad': 2, 'not_good': 15, 'total_bad': 49}
18


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f089ffd4d30>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.23723118212850308
sparse_theta_sp: -1.337108537108537
decorrelation: 0.01
ext_decorr_good: 100000
DOWNFALL: some good topics lost...
DOWNFALL: more bad topics...
num_topics: {'good': 4, 'bad': 4, 'not_good': 16, 'total_bad': 53}
19


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f0885f3abe0>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.22240423324547168
sparse_theta_sp: -1.2535392535392536
decorrelation: 0.01
ext_decorr_good: 100000
DOWNFALL: some good topics lost...
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 7, 'bad': 2, 'not_good': 13, 'total_bad': 55}


In [57]:
1

1

In [58]:
! ls results/postnauka

decorrelation.json	   iterative_10000.json       lda.json	   tless.json
iterative_100000.json	   iterative2_100000000.json  plsa.json
iterative_100000_new.json  iterative2_10000000.json   sparse.json


In [59]:
! mkdir -p results/postnauka/ablation_study

In [60]:
results.keys()

dict_keys([(0, 1, 1), (1, 0, 1), (1, 1, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1)])

In [64]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [66]:
'-'.join(str(i) for i in k)

'0-0-1'

In [67]:
for k, r in results.items():
    output_k = '-'.join(str(i) for i in k)
    with open(SAVE_FOLDER + f'/ablation_study/iterative_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [68]:
! ls results/postnauka/ablation_study

iterative_100000_0-0-1.json  iterative_100000_1-0-0.json
iterative_100000_0-1-0.json  iterative_100000_1-0-1.json
iterative_100000_0-1-1.json  iterative_100000_1-1-0.json


In [70]:
for k, r in results.items():
    print(k)
    print(r[-1]['scores'])
    print(r[-1]['num_topics'])
    print()

(0, 1, 1)
{'perplexity': 3312.350830078125, 'coherence_20': 0.9332480773125661, 'diversity_euclidean': 0.06855459069306795, 'diversity_jensenshannon': 0.7081095983377022, 'diversity_hellinger': 0.8284047549838643, 'diversity_cosine': 0.8665957850789532}
{'good': 10, 'bad': 4, 'not_good': 10, 'total_bad': 65}

(1, 0, 1)
{'perplexity': 3649.879638671875, 'coherence_20': 0.9451847342681219, 'diversity_euclidean': 0.08563056618044705, 'diversity_jensenshannon': 0.7174702594609959, 'diversity_hellinger': 0.8446588923717343, 'diversity_cosine': 0.8809849008195828}
{'good': 15, 'bad': 2, 'not_good': 5, 'total_bad': 40}

(1, 1, 0)
{'perplexity': 3749.14306640625, 'coherence_20': 1.0547720776041782, 'diversity_euclidean': 0.12004707522841225, 'diversity_jensenshannon': 0.7482016735133133, 'diversity_hellinger': 0.8676171608086275, 'diversity_cosine': 0.917459612167776}
{'good': 19, 'bad': 0, 'not_good': 1, 'total_bad': 19}

(1, 0, 0)
{'perplexity': 3557.381103515625, 'coherence_20': 0.904176757

In [71]:
! tail -n 50 results/postnauka/iterative_100000_new.json

            "17": 1.0524228260033137,
            "18": 1.0296176298910718,
            "19": 1.1678870612155294
        },
        "num_topics": {
            "good": 18,
            "bad": 0,
            "not_good": 2,
            "total_bad": 19
        }
    },
    {
        "scores": {
            "perplexity": 3789.63671875,
            "coherence_20": 1.0192302655028405,
            "diversity_euclidean": 0.27780393576980444,
            "diversity_jensenshannon": 0.7630736309319524,
            "diversity_hellinger": 0.9058558181488267,
            "diversity_cosine": 0.929081787382234
        },
        "topic_coherences": {
            "0": 0.5454667436822921,
            "1": 0.5754472410627138,
            "2": 1.302009849349046,
            "3": 0.9214287215829502,
            "4": 1.0884573939452766,
            "5": 1.0168754020127426,
            "6": 1.14367941540882,
            "7": 0.9419151443954846,
            "8": 0.9694617363799289,
            "9": 1.100954238

In [79]:
DECORRELATION_TAU = 100000000

ALL_PARAMS = [
    (0, 1, 1),
    (1, 0, 1),
    (1, 1, 0),

    (1, 0, 0),
    (0, 1, 0),
    (0, 0, 1),
]

In [80]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2

#  Best: 100000000.0 55.63102213541697
# Close: 10000000.0 55.890380859375

for params in ALL_PARAMS:
    key = params
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_TOPICS:
        print(seed)

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:
            custom_regularizers = dict()
            
            if params[0]:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)

            if params[1]:
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

(0, 1, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
decorrelation: 0.01
None
num_topics: {'good': 5, 'bad': 2, 'not_good': 15, 'total_bad': 2}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f0879f87d30>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f07d69324f0>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.23723118212850308
sparse_theta_sp: -1.337108537108537
decorrelation: 0.01
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: some good topics lost...
num_topics: {'good': 4, 'bad': 2, 'not_good': 16, 'total_bad': 4}
2


...

{'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f0823e2f3a0>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.3234970665388679
sparse_theta_sp: -1.823329823329823
decorrelation: 0.01
ext_decorr_good: 100000000
DOWNFALL: some good topics lost...
SUCCESS: less bad topics
num_topics: {'good': 4, 'bad': 2, 'not_good': 16, 'total_bad': 45}
18


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f088589e460>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.22240423324547168
sparse_theta_sp: -1.2535392535392536
decorrelation: 0.01
ext_decorr_good: 100000000
DOWNFALL: some good topics lost...
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 5, 'bad': 3, 'not_good': 15, 'total_bad': 48}
19


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f088591cb80>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.23723118212850308
sparse_theta_sp: -1.337108537108537
decorrelation: 0.01
ext_decorr_good: 100000000
DOWNFALL: some good topics lost...
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 7, 'bad': 2, 'not_good': 13, 'total_bad': 50}


In [81]:
1

1

In [82]:
! ls results/postnauka/ablation_study/

iterative_100000_0-0-1.json  iterative_100000_1-0-0.json
iterative_100000_0-1-0.json  iterative_100000_1-0-1.json
iterative_100000_0-1-1.json  iterative_100000_1-1-0.json


In [83]:
DECORRELATION_TAU

100000000

In [75]:
results.keys()

dict_keys([(0, 1, 1), (1, 0, 1), (1, 1, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1)])

In [84]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [85]:
for k, r in results.items():
    output_k = '-'.join(str(i) for i in k)
    with open(SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [86]:
! ls results/postnauka/ablation_study

iterative_100000_0-0-1.json  iterative2_100000000_0-0-1.json
iterative_100000_0-1-0.json  iterative2_100000000_0-1-0.json
iterative_100000_0-1-1.json  iterative2_100000000_0-1-1.json
iterative_100000_1-0-0.json  iterative2_100000000_1-0-0.json
iterative_100000_1-0-1.json  iterative2_100000000_1-0-1.json
iterative_100000_1-1-0.json  iterative2_100000000_1-1-0.json


In [87]:
for k, r in results.items():
    print(k)
    print(r[-1]['scores'])
    print(r[-1]['num_topics'])
    print()

(0, 1, 1)
{'perplexity': 3289.373779296875, 'coherence_20': 0.8988719308266891, 'diversity_euclidean': 0.061550219116740953, 'diversity_jensenshannon': 0.6805642620882165, 'diversity_hellinger': 0.7930348262974288, 'diversity_cosine': 0.8355798388347799}
{'good': 10, 'bad': 2, 'not_good': 10, 'total_bad': 61}

(1, 0, 1)
{'perplexity': 3596.740966796875, 'coherence_20': 0.9133727698674099, 'diversity_euclidean': 0.07424026965623534, 'diversity_jensenshannon': 0.6999792171054902, 'diversity_hellinger': 0.8227138971994776, 'diversity_cosine': 0.8522543405581078}
{'good': 13, 'bad': 3, 'not_good': 7, 'total_bad': 44}

(1, 1, 0)
{'perplexity': 3735.0888671875, 'coherence_20': 1.0339721589199644, 'diversity_euclidean': 0.12029459247713288, 'diversity_jensenshannon': 0.7332457577090314, 'diversity_hellinger': 0.8498986922194876, 'diversity_cosine': 0.8961662609427321}
{'good': 19, 'bad': 0, 'not_good': 1, 'total_bad': 19}

(1, 0, 0)
{'perplexity': 3557.381103515625, 'coherence_20': 0.90417675

In [88]:
! tail -n 50 results/postnauka/iterative2_100000000.json

            "17": 0.9970727934745751,
            "18": 1.0472985496415317,
            "19": 1.1678870612155292
        },
        "num_topics": {
            "good": 18,
            "bad": 0,
            "not_good": 2,
            "total_bad": 18
        }
    },
    {
        "scores": {
            "perplexity": 3749.740966796875,
            "coherence_20": 1.0035540046975566,
            "diversity_euclidean": 0.17922226431951543,
            "diversity_jensenshannon": 0.7398631312137611,
            "diversity_hellinger": 0.8581046722219192,
            "diversity_cosine": 0.9074182569493797
        },
        "topic_coherences": {
            "0": 0.9206962025174021,
            "1": 0.5766228619158285,
            "2": 1.302009849349046,
            "3": 1.0097633592414297,
            "4": 1.0884573939452766,
            "5": 1.0168754020127426,
            "6": 0.924416732773585,
            "7": 0.6151009034375062,
            "8": 0.9170213575321767,
            "9": 0.917